[Énoncés : Examen blanc B — Tableaux et logique](../examen_blanc_B.ipynb)

# Corrigé — Examen blanc B — Tableaux et logique

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec tous les tests (y compris les cas cachés de la correction). Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 54 — Alternance de signes

**Idée** : même schéma que l'exercice 52 : un compteur `courant` pour la portion qui se termine ici. Il augmente si le signe change par rapport au précédent, repart à 1 sinon, et tombe à 0 sur un zéro. `(x > 0) != (precedent > 0)` teste « signes opposés » pour des nombres non nuls.

**Complexité** : O(n).

In [ ]:
def alternance_signes(nombres: list[int]) -> int:
    meilleur = 0
    courant = 0                   # longueur de la portion alternée qui finit ici
    for i, x in enumerate(nombres):
        if x == 0:
            courant = 0           # un zéro casse tout
        elif courant > 0 and (x > 0) != (nombres[i - 1] > 0):
            courant += 1          # signe opposé au précédent : la portion continue
        else:
            courant = 1           # nouvelle portion qui commence ici
        meilleur = max(meilleur, courant)
    return meilleur


verifier(alternance_signes, [
    (([-1, 2, -3, 4, 0, -1, 2],), 4),
    (([1, 1, 1],), 1),
    (([],), 0),
    (([0, 0],), 0),   # que des zéros
    (([5, -5],), 2),
    (([1, -1, 1, 1, -1, 1, -1],), 4),
    (([-2, 0, 3],), 1),
])

## Exercice 55 — Centre de croix le plus lumineux

**Idée** : ne parcourir que les cases intérieures (`range(1, len - 1)`), ce qui garantit que les quatre voisins existent : pas besoin de tester les bornes. Comparaison stricte pour respecter « plus haute, puis plus à gauche ».

**Complexité** : O(lignes × colonnes).

In [ ]:
def croix_plus_forte(grille: list[list[int]]) -> list[int]:
    meilleur = None
    for i in range(1, len(grille) - 1):              # seulement les cases intérieures
        for j in range(1, len(grille[0]) - 1):
            lum = (grille[i][j] + grille[i - 1][j] + grille[i + 1][j]
                   + grille[i][j - 1] + grille[i][j + 1])
            if meilleur is None or lum > meilleur[0]:   # > strict : en cas d'égalité, la première trouvée
                meilleur = (lum, i, j)
    return [meilleur[1], meilleur[2]]


verifier(croix_plus_forte, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [1, 1]),
    (([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 2, 9], [0, 0, 0, 0]],), [2, 2]),
    (([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]],), [1, 1]),   # égalité partout
    (([[1, 1, 1, 1], [1, 9, 1, 1], [1, 1, 1, 1], [1, 1, 5, 1]],), [1, 1]),   # égalité : la plus haute
    (([[5, 5, 5], [5, 0, 5], [5, 5, 5]],), [1, 1]),
    (([[-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1]],), [1, 1]),   # valeurs négatives
])

## Exercice 56 — Paires de valeurs séparées par un écart

**Idée** : comme on compte des paires de **valeurs**, on passe par un **set** (les doublons disparaissent). Pour chaque valeur `x`, il y a au plus un partenaire possible, `x + ecart`, et le test `in` sur un set est en O(1). Autre solution O(n) sans set : deux pointeurs qui avancent dans le même sens sur la liste triée, en sautant les doublons.

**Complexité** : O(n).

In [ ]:
def paires_ecart(nombres: list[int], ecart: int) -> int:
    valeurs = set(nombres)                                  # les doublons disparaissent
    return sum(1 for x in valeurs if x + ecart in valeurs)  # chaque x a au plus un partenaire y = x + ecart


verifier(paires_ecart, [
    (([1, 1, 2, 3, 4, 4], 2), 2),
    (([], 1), 0),
    (([1, 2, 3], 5), 0),
    (([1, 2, 3, 4], 1), 3),
    (([5, 5, 5, 6], 1), 1),   # les doublons ne comptent qu'une fois
    (([-3, -1, 1, 3], 2), 3),
    (([1, 3, 3, 5, 5, 7], 2), 3),
])

In [ ]:
# Test de vitesse
chrono(paires_ecart, [i // 2 for i in range(20_000)], 3, attendu=9997)
chrono(paires_ecart, list(range(0, 60_000, 3)), 6, attendu=19998)
chrono(paires_ecart, list(range(20_000)), 20_000, attendu=0)
chrono(paires_ecart, list(range(0, 40_000, 2)), 1, attendu=0)

## Exercice 57 — Ordre des tickets urgents

**Idée** : un tri avec une clé tuple : `(-urgence, identifiant)`. Le signe `-` inverse l'ordre de l'urgence (décroissant), l'identifiant départage les égalités (croissant). C'est la même technique qu'aux exercices 6 et 34.

**Complexité** : O(n log n).

In [ ]:
def tickets_urgents(tickets: list[list[int]]) -> list[int]:
    ordre = sorted(tickets, key=lambda t: (-t[1], t[0]))     # urgence ↓, puis identifiant ↑
    return [identifiant for identifiant, urgence in ordre]


verifier(tickets_urgents, [
    (([[8, 2], [3, 5], [7, 2]],), [3, 7, 8]),
    (([],), []),
    (([[1, 1]],), [1]),
    (([[5, 1], [4, 1], [3, 1]],), [3, 4, 5]),   # même urgence : par identifiant
    (([[1, 1], [2, 2], [3, 3]],), [3, 2, 1]),
    (([[10, 0], [2, 9], [7, 9], [1, 0]],), [2, 7, 1, 10]),
])

## Exercice 58 — Escalier avec pas de 1, 2 ou 3

**Idée** : programmation dynamique. Pour arriver à la marche `m`, le dernier pas vient de `m - 1`, `m - 2` ou `m - 3` : `facons[m] = facons[m-1] + facons[m-2] + facons[m-3]`. La version récursive directe recalcule sans cesse les mêmes valeurs et devient exponentielle ; le tableau calcule chaque valeur une seule fois.

**Complexité** : O(n).

In [ ]:
def facon_monter(n: int) -> int:
    # facons[m] = nombre de façons d'arriver exactement à la marche m
    facons = [0] * (n + 1)
    facons[0] = 1
    for m in range(1, n + 1):
        for pas in (1, 2, 3):
            if m - pas >= 0:
                facons[m] += facons[m - pas]     # le dernier pas a été de 1, 2 ou 3 marches
    return facons[n]


verifier(facon_monter, [
    ((3,), 4),
    ((4,), 7),
    ((0,), 1),
    ((1,), 1),
    ((2,), 2),
    ((5,), 13),
    ((10,), 274),
    ((20,), 121415),
])

---

[Énoncés de l'examen](../examen_blanc_B.ipynb)